# ABC Bank CLU - Create and Train

This notebook:
- creates or reuses a **Conversational Language Understanding** project
- imports intents, entities, and labeled utterances
- configures `account_type` as a learned entity
- configures `amount` as a learned entity with the prebuilt `Quantity.Currency` component
- starts **Standard** training with an **80/20 automatic split**

In [ ]:
# %pip install python-dotenv requests pandas

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import HttpResponseError
from azure.ai.language.conversations.authoring import ConversationAuthoringClient
from azure.ai.language.conversations.authoring.models import (
    CreateProjectOptions,
    ProjectKind,
    ExportedProject,
    ConversationExportedProjectAsset,
    ProjectSettings,
    ConversationExportedIntent,
    ConversationExportedEntity,
    ConversationExportedUtterance,
    ExportedUtteranceEntityLabel,
    ExportedPrebuiltEntity,
    TrainingJobDetails,
    TrainingMode,
    EvaluationDetails,
    EvaluationKind,
)
import os, json

In [ ]:
load_dotenv(".env")

LANGUAGE_ENDPOINT = https://REDACTED_ENDPOINT/
LANGUAGE_KEY = REDACTED_AZURE_KEY

if not LANGUAGE_ENDPOINT or not LANGUAGE_KEY:
    raise ValueError("Missing AZURE_LANGUAGE_ENDPOINT / AZURE_LANGUAGE_KEY (or CQA equivalents) in .env")

PROJECT_NAME = "sample-resource-name"
PROJECT_LANGUAGE = "en-us"
PROJECT_DESCRIPTION = "Customer Service Assistance for ABC Bank"
MODEL_LABEL = "abc-bank-clu-model-v1"

client = ConversationAuthoringClient(
    endpoint=LANGUAGE_ENDPOINT,
    credential=AzureKeyCredential(LANGUAGE_KEY),
)

print("ConversationAuthoringClient ready.")
print("Endpoint:", LANGUAGE_ENDPOINT)
print("Project Name:", PROJECT_NAME)
print("Model Label:", MODEL_LABEL)

In [ ]:
## ----- HELPERS -----
def upsert_env_file(path: Path, updates: dict):
    existing = {}

    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            existing[k] = v.strip().strip('"')

    existing.update(updates)

    with path.open("w", encoding="utf-8") as f:
        for k, v in existing.items():
            f.write(f'{k}="{v}"\n')


def ensure_project():
    try:
        details = client.get_project(project_name=PROJECT_NAME)
        print(f"Project already exists: {details.project_name}")
        return details
    except HttpResponseError as e:
        if getattr(e, "status_code", None) != 404:
            raise

    body = CreateProjectOptions(
        project_kind=ProjectKind.CONVERSATION,
        project_name=PROJECT_NAME,
        language=PROJECT_LANGUAGE,
        multilingual=False,
        description=PROJECT_DESCRIPTION,
    )

    created = client.create_project(project_name=PROJECT_NAME, body=body)
    print(f"Created project: {created.project_name}")
    return created


def label_span(text: str, phrase: str, category: str):
    start = text.lower().index(phrase.lower())
    return ExportedUtteranceEntityLabel(
        category=category,
        offset=start,
        length=len(phrase),
    )


def utterance(text: str, intent: str, labels=None, dataset="Train", language=PROJECT_LANGUAGE):
    return ConversationExportedUtterance(
        text=text,
        language=language,
        dataset=dataset,
        intent=intent,
        entities=labels or [],
    )

In [ ]:
## ----- ENSURE PROJECT -----
project = ensure_project()

upsert_env_file(Path(".env"), {
    "AZURE_CLU_PROJECT_NAME": PROJECT_NAME,
    "AZURE_CLU_PROJECT_LANGUAGE": PROJECT_LANGUAGE,
    "AZURE_CLU_MODEL_LABEL": MODEL_LABEL,
})

print("Saved CLU settings to .env")

In [ ]:
# =========================
# DEFINE INTENTS
# =========================
intents = [
    ConversationExportedIntent(category="CheckBalance"),
    ConversationExportedIntent(category="TransferFunds"),
    ConversationExportedIntent(category="LoanStatus"),
]

# =========================
# DEFINE ENTITIES
# =========================
entities = [
    ConversationExportedEntity(
        category="account_type",
        composition_mode="combineComponents",
    ),
    ConversationExportedEntity(
        category="amount",
        composition_mode="combineComponents",
        prebuilts=[
            ExportedPrebuiltEntity(category="Quantity.Currency")
        ],
        required_components=["learned", "prebuilts"],
    ),
]

print("Defined intents and entities.")

In [ ]:
# =========================
# BUILD UTTERANCES
# =========================
utterances = []

# Intent 1 – CheckBalance
text = "What’s my current balance in the savings account?"
utterances.append(utterance(text, "CheckBalance", [
    label_span(text, "savings account", "account_type")
]))

text = "How much do I have in my fixed deposit account?"
utterances.append(utterance(text, "CheckBalance", [
    label_span(text, "fixed deposit account", "account_type")
]))

text = "Can you tell me my current account balance?"
utterances.append(utterance(text, "CheckBalance", [
     label_span(text, "current account", "account_type")
]))

text = "Show me the balance in my savings account."
utterances.append(utterance(text, "CheckBalance", [
    label_span(text, "savings account", "account_type")
]))

text = "What is the balance in my loan account?"
utterances.append(utterance(text, "CheckBalance", [
    label_span(text, "loan account", "account_type")
]))

text = "How much money do I have in my savings account ?"
utterances.append(utterance(text, "CheckBalance", [
    label_span(text, "savings account", "account_type")
]))

# Intent 2 – TransferFunds
text = "Transfer $5000 from my savings to fixed deposit account."
utterances.append(utterance(text, "TransferFunds", [
    label_span(text, "$5000", "amount"),
    label_span(text, "savings", "account_type"),
    label_span(text, "fixed deposit account", "account_type"),
]))

text = "Move $2000 to my current account from savings."
utterances.append(utterance(text, "TransferFunds", [
    label_span(text, "$2000", "amount"),
    label_span(text, "current account", "account_type"),
    label_span(text, "savings", "account_type"),
]))

text = "Can you transfer $300 to my savings account?"
utterances.append(utterance(text, "TransferFunds", [
    label_span(text, "$300", "amount"),
    label_span(text, "savings account", "account_type"),
]))

text = "Send $1000 from my savings to loan account."
utterances.append(utterance(text, "TransferFunds", [
    label_span(text, "$1000", "amount"),
    label_span(text, "savings", "account_type"),
    label_span(text, "loan account", "account_type"),
]))

text = "Please transfer $500 to my loan account from savings."
utterances.append(utterance(text, "TransferFunds", [
    label_span(text, "$500", "amount"),
    label_span(text, "loan account", "account_type"),
    label_span(text, "savings", "account_type"),
]))

text = "Move funds to savings."
utterances.append(utterance(text, "TransferFunds", [
    label_span(text, "savings", "account_type")
]))

# Intent 3 – LoanStatus
text = "What’s the status of my loan application?"
utterances.append(utterance(text, "LoanStatus", []))

text = "Has my home loan been approved yet?"
utterances.append(utterance(text, "LoanStatus", []))

text = "When will I get an update on my loan status?"
utterances.append(utterance(text, "LoanStatus", []))

text = "Is my loan application under review?"
utterances.append(utterance(text, "LoanStatus", []))

text = "Can you tell me if my personal loan was approved?"
utterances.append(utterance(text, "LoanStatus", []))

text = "How’s my car loan application going?"
utterances.append(utterance(text, "LoanStatus", []))

print(f"Built {len(utterances)} utterances.")

In [ ]:
# =========================
# BUILD IMPORT PAYLOAD
# =========================
exported_project = ExportedProject(
    project_file_version="2025-11-15-preview",
    string_index_type="Utf16CodeUnit",
    metadata=CreateProjectOptions(
        project_kind=ProjectKind.CONVERSATION,
        project_name=PROJECT_NAME,
        language=PROJECT_LANGUAGE,
        multilingual=False,
        description=PROJECT_DESCRIPTION,
        settings=ProjectSettings(confidence_threshold=0.0),
    ),
    assets=ConversationExportedProjectAsset(
        project_kind=ProjectKind.CONVERSATION,
        intents=intents,
        entities=entities,
        utterances=utterances,
    ),
)

print("Payload ready.")
print("Entities configured:")
for ent in entities:
    print(" -", ent.category)

In [ ]:
# =========================
# IMPORT / OVERWRITE PROJECT ASSETS
# =========================
project_client = client.get_project_client(PROJECT_NAME)

poller = project_client.project.begin_import(
    body=exported_project,
    exported_project_format="Conversation",
)

import_result = poller.result()

print("Import completed.")
print(import_result)

In [ ]:
# =========================
# TRAIN MODEL
# =========================
training_body = TrainingJobDetails(
    model_label=MODEL_LABEL,
    training_mode=TrainingMode.STANDARD,
    evaluation_options=EvaluationDetails(
        kind=EvaluationKind.PERCENTAGE,
        training_split_percentage=80,
        testing_split_percentage=20,
    ),
    # training_config_version omitted intentionally
    # service uses latest by default
)

poller = project_client.project.begin_train(body=training_body)
train_result = poller.result()

print("Training completed.")
print(train_result)

In [ ]:
# =========================
# LIST TRAINED MODELS
# =========================
models = list(project_client.project.list_trained_models())

print("Trained models:")
for m in models:
    print(
        " -",
        getattr(m, "model_label", None)
        or getattr(m, "trained_model_label", None)
        or m
    )